# CredResolve Collections Analytics — Full Analysis
Investigates whether the reported **11% MoM recovery improvement** is real, why performance changed, and where the ₹10 Cr investment should go.

Reasoning is shown at each step — see `docs/DATA_QUALITY_REPORT.md` and `docs/EXECUTIVE_MEMO.md` for the condensed findings.

## Part 1: Initial Profiling
Start by checking basic integrity: row counts, exact duplicates, and primary-key cardinality for every table, before trusting anything.

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DATA = '/home/claude/credresolve/data'
tables = [f.replace('.csv','') for f in os.listdir(DATA) if f.endswith('.csv') and f != 'data_dictionary.csv']

dfs = {}
for t in tables:
    dfs[t] = pd.read_csv(f'{DATA}/{t}.csv')

print("="*100)
print("TABLE SHAPES & EXACT DUPLICATE ROWS")
print("="*100)
for t, df in sorted(dfs.items()):
    dupe_full = df.duplicated().sum()
    print(f"{t:28s} rows={len(df):7d}  cols={len(df.columns):3d}  exact_dupe_rows={dupe_full:6d}")

print()
print("="*100)
print("PRIMARY-KEY-LIKE COLUMN DUPLICATE CHECK (first column of each table)")
print("="*100)
for t, df in sorted(dfs.items()):
    pk = df.columns[0]
    n_total = len(df)
    n_unique = df[pk].nunique()
    print(f"{t:28s} pk={pk:20s} total={n_total:7d}  unique={n_unique:7d}  dupe_pk_rows={n_total-n_unique:6d}")


## Agent & Borrower Identity Investigation
`agents.csv` shows only 1,000 distinct `agent_id`s across 30,000 rows (avg 30 rows/agent). Before assuming this is a normal dimension table, check whether the non-key attributes (name, team, vendor, status) are actually *stable* per ID.

In [ ]:
import pandas as pd
agents = pd.read_csv('data/agents.csv')
sample_id = agents['agent_id'].value_counts().index[0]
print(agents[agents.agent_id==sample_id][['agent_id','employee_code','agent_name','vendor_id','team','status']].head(10))
print()
print("Distinct agent_name values in the WHOLE file:", agents.agent_name.nunique(), "(out of", len(agents), "rows)")

**Finding:** for a single `agent_id`, name/team/vendor/status are all different on almost every row, and only 10 distinct names exist across the entire file. This isn't 'same agent, multiple IDs' — it's the ID and its attributes being essentially decoupled. Confirmed the same pattern in `borrowers.csv` (phone/email/created_at also vary per `borrower_id`). **Decision: exclude these dimension tables' descriptive attributes from all downstream analysis.**

## Timezone / Time-of-Day Investigation
The assignment specifically flags 'calls being classified into the wrong hour/day' as a risk to check.

In [ ]:
import pandas as pd
calls = pd.read_csv('data/calls.csv', parse_dates=['event_at'])
calls['hour'] = calls.event_at.dt.hour
pivot = calls.groupby(['timezone','hour']).size().unstack(0)
pivot_pct = (pivot / pivot.sum() * 100)
print(pivot_pct.round(2))
print()
print("Correlation between hourly distributions across stated timezones:")
print(pivot_pct.corr())

**Finding:** hour-of-day is flat (~4.17%, i.e. uniform) for every stated timezone, with near-zero correlation between them. Sanity-checked against `agent_sessions.login_at` — same flat pattern. **There is no genuine time-of-day signal in this dataset.** Any 'optimal calling time' conclusion would be manufactured, not discovered — flagged as a hard limitation rather than forced.

## Part 3: Is the 11% Improvement Real?
Build the golden (deduplicated) payments table and compare month-on-month trends under both the naive (as-likely-reported) and cleaned definitions.

In [ ]:
"""
GOLDEN DATASET PIPELINE — CredResolve Collections Analytics
==============================================================
Documents every cleaning decision inline. Run: python3 02_golden_dataset.py
Outputs golden tables to /home/claude/credresolve/outputs/golden/
"""
import pandas as pd
import numpy as np
import os

DATA = '/home/claude/credresolve/data'
OUT = '/home/claude/credresolve/outputs/golden'
os.makedirs(OUT, exist_ok=True)

log = []
def report(step, raw, kept, note=""):
    rejected = raw - kept
    pct = rejected/raw*100 if raw else 0
    log.append((step, raw, kept, rejected, f"{pct:.2f}%", note))
    print(f"{step:45s} raw={raw:8d}  kept={kept:8d}  rejected={rejected:7d} ({pct:5.2f}%)  {note}")

# ------------------------------------------------------------------
# 1. ACCOUNT SPINE — accounts.csv is the canonical entity table.
#    Verified: account_id has zero duplicates, zero exact-dupe rows.
#    Decision: this is our SOURCE OF TRUTH for account_id, borrower_id (FK),
#    loan_type, principal, outstanding, dpd, risk_segment, opened_at.
# ------------------------------------------------------------------
accounts = pd.read_csv(f'{DATA}/accounts.csv', parse_dates=['opened_at'])
raw_n = len(accounts)
# 455 accounts have null borrower_id — kept, but flagged as unresolved-borrower accounts
# (excluding them would silently shrink the population — a denominator-manipulation risk
# the assignment specifically warns against, so we keep them and flag instead)
accounts['borrower_resolved'] = accounts['borrower_id'].notna()
report("accounts (spine)", raw_n, len(accounts),
       f"{(~accounts.borrower_resolved).sum()} accounts flagged: no borrower_id")

# ------------------------------------------------------------------
# 2. POINT-IN-TIME ACCOUNT STATUS — account_status_history.csv
#    Decision: event_at is the canonical timestamp for status changes.
#    recorded_at is NOT used for ordering — 50.3% of rows have recorded_at
#    BEFORE event_at (impossible for real ingestion), consistent with a
#    uniform random ±24h jitter rather than genuine late-arrival signal.
#    This makes recorded_at unusable for "as of ingestion" logic; event_at
#    is treated as ground truth for when the status change happened.
# ------------------------------------------------------------------
hist = pd.read_csv(f'{DATA}/account_status_history.csv', parse_dates=['event_at','recorded_at'])
raw_n = len(hist)
hist_dedup = hist.drop_duplicates(subset=['account_id','event_at','status'])
report("account_status_history", raw_n, len(hist_dedup), "exact (account,event_at,status) dupes dropped")

def status_as_of(account_ids, as_of_date, hist_df, accounts_df):
    """Reconstruct account status as of a given date using event_at ordering."""
    h = hist_df[hist_df.event_at <= as_of_date].sort_values('event_at')
    latest = h.groupby('account_id').tail(1).set_index('account_id')['status']
    result = latest.reindex(account_ids)
    # accounts with NO history on/before as_of_date fall back to accounts.csv's
    # current status field (documented fallback — see notes)
    fallback = accounts_df.set_index('account_id')['status']
    result = result.fillna(fallback)
    return result

n_no_history = accounts.account_id.nunique() - hist_dedup.account_id.nunique()
print(f"  -> {n_no_history} accounts have ZERO status-history rows; fallback = accounts.csv current status field")

# ------------------------------------------------------------------
# 3. PAYMENTS — the highest-stakes cleaning decision in this pipeline.
#    Two distinct duplication patterns found:
#      (a) Exact full-row duplicates (972 rows / 485 references) — pure
#          re-ingestion noise, e.g. a retry-safe API call inserted twice.
#      (b) Same payment_reference, different payment_id, MULTIPLE SUCCESS
#          rows (2,033 references) — could be legit retry-after-failure,
#          but per-reference multiple SUCCESS rows cannot both be real
#          money (a payment reference is issued once per transaction
#          attempt in every payment gateway design).
#    Decision: for each payment_reference, keep exactly ONE row —
#      - if any SUCCESS rows exist for that reference, keep the
#        EARLIEST SUCCESS (first successful settlement; later "SUCCESS"
#        rows for the same reference are treated as duplicate settlement
#        confirmations, a known payment-gateway callback-retry pattern)
#      - if no SUCCESS rows exist, keep the latest row chronologically
#        (most recent status is most informative for FAILED/PENDING/REVERSED)
#    This is a MATERIAL decision — it changes total recovery by ~14%.
#    Documented explicitly so it can be challenged/changed downstream.
# ------------------------------------------------------------------
payments = pd.read_csv(f'{DATA}/payments.csv', parse_dates=['event_at'])
raw_n = len(payments)

succ = payments[payments.payment_status == 'SUCCESS'].sort_values('event_at')
succ_dedup = succ.drop_duplicates(subset='payment_reference', keep='first')

non_succ = payments[payments.payment_status != 'SUCCESS']
refs_with_success = set(succ.payment_reference)
non_succ_no_success = non_succ[~non_succ.payment_reference.isin(refs_with_success)]
non_succ_dedup = non_succ_no_success.sort_values('event_at').drop_duplicates(subset='payment_reference', keep='last')

payments_golden = pd.concat([succ_dedup, non_succ_dedup], ignore_index=True)
report("payments", raw_n, len(payments_golden),
       f"amount impact: naive SUCCESS sum ={payments[payments.payment_status=='SUCCESS'].amount.sum():,.0f} "
       f"-> golden ={payments_golden[payments_golden.payment_status=='SUCCESS'].amount.sum():,.0f}")

# ------------------------------------------------------------------
# 4. CALLS — drop exact duplicate rows (1,271 found) and duplicate
#    call_id with differing content (1,350 call_ids appear >1x total;
#    keep first occurrence chronologically per call_id).
# ------------------------------------------------------------------
calls = pd.read_csv(f'{DATA}/calls.csv', parse_dates=['event_at'])
raw_n = len(calls)
calls_dedup = calls.sort_values('event_at').drop_duplicates(subset='call_id', keep='first')
report("calls", raw_n, len(calls_dedup), "dedup by call_id, keep earliest event_at")

# ------------------------------------------------------------------
# 5. CALL DISPOSITIONS — normalize legacy synonym codes.
#    PROMISE_TO_PAY and PTP co-occur across ALL disposition_version
#    values at similar rates -> not a version-driven rename, but a
#    genuine duplicate label for the same outcome. Canonicalized to PTP.
# ------------------------------------------------------------------
disp = pd.read_csv(f'{DATA}/call_dispositions.csv', parse_dates=['event_at'])
raw_n = len(disp)
code_map = {'PROMISE_TO_PAY': 'PTP'}
disp['disposition_code_clean'] = disp['disposition_code'].replace(code_map)
disp_dedup = disp.drop_duplicates(subset=['call_id','disposition_code_clean'])
report("call_dispositions", raw_n, len(disp_dedup),
       f"{(disp.disposition_code=='PROMISE_TO_PAY').sum()} PROMISE_TO_PAY rows merged into PTP")

# ------------------------------------------------------------------
# 6. WHATSAPP EVENTS / BORROWERS — drop exact duplicate rows only.
#    Borrower demographic fields (name/phone/email/city/state) are NOT
#    used downstream — see data-quality report: these attributes are
#    statistically decoupled from borrower_id and cannot be trusted for
#    segmentation. borrower_id is retained ONLY as an FK linking accounts.
# ------------------------------------------------------------------
wa = pd.read_csv(f'{DATA}/whatsapp_events.csv')
wa_dedup = wa.drop_duplicates()
report("whatsapp_events", len(wa), len(wa_dedup), "exact dupes dropped")

borrowers_raw = pd.read_csv(f'{DATA}/borrowers.csv')
report("borrowers (dimension — DO NOT TRUST attributes)", len(borrowers_raw),
       borrowers_raw.borrower_id.nunique(),
       "collapsed to distinct borrower_id only; name/phone/email/city/state EXCLUDED from analysis")

# ------------------------------------------------------------------
# SAVE GOLDEN TABLES
# ------------------------------------------------------------------
accounts.to_csv(f'{OUT}/accounts_golden.csv', index=False)
payments_golden.to_csv(f'{OUT}/payments_golden.csv', index=False)
calls_dedup.to_csv(f'{OUT}/calls_golden.csv', index=False)
disp_dedup.to_csv(f'{OUT}/dispositions_golden.csv', index=False)
hist_dedup.to_csv(f'{OUT}/status_history_golden.csv', index=False)

log_df = pd.DataFrame(log, columns=['step','raw_rows','kept_rows','rejected_rows','rejected_pct','note'])
log_df.to_csv(f'{OUT}/cleaning_impact_log.csv', index=False)
print("\nGolden tables written to", OUT)


In [ ]:
import pandas as pd
raw_payments = pd.read_csv('data/payments.csv', parse_dates=['event_at'])
golden_payments = pd.read_csv('outputs/golden/payments_golden.csv', parse_dates=['event_at'])

def monthly_recovery(df):
    s = df[df.payment_status=='SUCCESS'].copy()
    s['month'] = s.event_at.dt.to_period('M')
    return s.groupby('month')['amount'].sum()

naive = monthly_recovery(raw_payments)
golden = monthly_recovery(golden_payments)
print("NAIVE MoM %:\n", naive.pct_change().round(4)*100)
print("\nGOLDEN MoM %:\n", golden.pct_change().round(4)*100)
print("\nFeb->Mar naive MoM:", round(naive.pct_change()['2026-03']*100,2), "% <- matches the reported '11%' almost exactly")

**Finding:** the reported 11% is a real number for exactly one month pair (Feb→Mar, naive), presented as if it were a sustained trend. The actual 7-month trend (excluding the truncated partial August) is flat under the naive definition (-0.45% overall) and **declining 18.6%** under the deduplicated definition. This is the central finding of the assignment.

## Part 2: Driver Analysis
Test every driver leadership asked about that the data can actually support.

In [ ]:
import pandas as pd
pd.set_option('display.width', 160)

DATA = '/home/claude/credresolve/data'
GOLD = '/home/claude/credresolve/outputs/golden'

accounts = pd.read_csv(f'{DATA}/accounts.csv')
calls = pd.read_csv(f'{GOLD}/calls_golden.csv')
attempts = pd.read_csv(f'{DATA}/call_attempts.csv')
payments = pd.read_csv(f'{GOLD}/payments_golden.csv')
vendor_t = pd.read_csv(f'{DATA}/vendor_telephony.csv')
campaigns = pd.read_csv(f'{DATA}/campaigns.csv')

acc = accounts[['account_id','risk_segment','dpd','loan_type']].copy()
acc['dpd_bucket'] = pd.cut(acc.dpd, [0,15,30,60,90,9999], labels=['0-15','16-30','31-60','61-90','90+'])

print("="*90); print("DRIVER: RISK SEGMENT — contact rate & recovery"); print("="*90)
c = calls.merge(acc, on='account_id', how='left')
print("Contact rate (ANSWERED %) by risk_segment:")
print(c.groupby('risk_segment', observed=True).apply(lambda g: (g.call_status=='ANSWERED').mean()*100, include_groups=False).round(2))
p = payments[payments.payment_status=='SUCCESS'].merge(acc, on='account_id', how='left')
print("\nTotal recovered by risk_segment:")
print(p.groupby('risk_segment', observed=True)['amount'].sum().round(0))

print()
print("="*90); print("DRIVER: DPD BUCKET"); print("="*90)
print("Contact rate by dpd_bucket:")
print(c.groupby('dpd_bucket', observed=True).apply(lambda g: (g.call_status=='ANSWERED').mean()*100, include_groups=False).round(2))
print("\nRecovered amount by dpd_bucket:")
print(p.groupby('dpd_bucket', observed=True)['amount'].sum().round(0))

print()
print("="*90); print("DRIVER: TELEPHONY VENDOR (grouped by real vendor_name, not vendor_id)"); print("="*90)
c2 = calls.merge(vendor_t[['vendor_id','vendor_name']], on='vendor_id', how='left')
print(c2.groupby('vendor_name').apply(lambda g: (g.call_status=='ANSWERED').mean()*100, include_groups=False).round(2))

print()
print("="*90); print("DRIVER: CAMPAIGN CHANNEL"); print("="*90)
c3 = calls.merge(campaigns[['campaign_id','channel','strategy_version']], on='campaign_id', how='left')
print("Contact rate by campaign channel:")
print(c3.groupby('channel').apply(lambda g: (g.call_status=='ANSWERED').mean()*100, include_groups=False).round(2))
print("\nContact rate by strategy_version:")
print(c3.groupby('strategy_version').apply(lambda g: (g.call_status=='ANSWERED').mean()*100, include_groups=False).round(2))

print()
print("="*90); print("DRIVER: ATTEMPT FREQUENCY (attempt_no) — does calling more help or hurt?"); print("="*90)
conn_rate = attempts.groupby('attempt_no').apply(lambda g: (g.attempt_status=='CONNECTED').mean()*100, include_groups=False)
print(conn_rate.round(2))


**Finding:** none of risk segment, DPD, telephony vendor, campaign channel, strategy version, or attempt frequency show a meaningful effect on contact rate or recovery rate — all cluster within a ~1 percentage point band. Geography, language, and agent tenure could not be tested (unreliable dimension tables). This is reported as a genuine negative finding.

## PTP-to-Cash Reality Check
Given contact/recovery rates are flat but recovery-per-account is declining, check whether 'kept' promises-to-pay actually correspond to real payments.

In [ ]:
import pandas as pd
ptp = pd.read_csv('data/promises_to_pay.csv', parse_dates=['event_at','promised_date'])
payments = pd.read_csv('outputs/golden/payments_golden.csv', parse_dates=['event_at'])
succ_pay = payments[payments.payment_status=='SUCCESS'][['account_id','event_at','amount']]

kept = ptp[ptp.status=='KEPT']
accts_with_kept_ptp = set(kept.account_id)
accts_with_any_payment = set(succ_pay.account_id)
overlap = accts_with_kept_ptp & accts_with_any_payment
print(f"Accounts with a KEPT PTP: {len(accts_with_kept_ptp)}")
print(f"Of those, with ANY successful payment ever, any date: {len(overlap)} ({len(overlap)/len(accts_with_kept_ptp)*100:.1f}%)")

**Finding:** only 41.2% of accounts with a 'KEPT' PTP have any successful payment ever recorded — even with an unbounded time window. This is a genuine data-integrity gap between the `promises_to_pay.status` field and actual cash, and a plausible contributor to the recovery-per-account decline (PTP-kept-rate looks stable, but isn't validated against real repayment).

## Part 4: Investment Recommendation
See full reasoning and numbers in `docs/EXECUTIVE_MEMO.md`. Summary: WhatsApp/Digital Engagement shows the best return-per-rupee among tested channels (comparable ₹/touch to expensive channels at a fraction of the cost), but 85% of successful payments have no attributable touchpoint in any channel — so we recommend a capped ₹1–1.5 Cr randomized holdout pilot before committing the full ₹10 Cr.

## Conclusion
1. **The 11% claim is false as a trend** — it's a single-month artifact.
2. **True trend is flat-to-declining**, most clearly seen in recovery-per-account and recovery-per-agent-hour (-19%).
3. **No standard operational driver explains it** — the funnel's top (contact) is stable; something downstream (plausibly PTP-to-cash conversion) is degrading.
4. **Data quality itself is the headline story**: agent/borrower identity, timestamps, and PTP status are all unreliable in ways that would silently corrupt any naive analysis.
5. **Investment recommendation is conditional**: fund a pilot before a full ₹10 Cr commit, given the attribution gap.